# Data Overview

This notebook explores the raw Lending Club loan dataset to understand its structure, key fields, missing values, and potential signals related to loan repayment risk.


## Raw Dataset Inspection

The raw dataset contains one or more CSV files representing historical loan records. Initial inspection focuses on understanding file size, available columns, and potential label fields without loading the entire dataset into memory.


In [11]:
import pandas as pd

pd.set_option('display.max_columns', None)

DATA_RAW_PATH = "../data/raw/archive/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"
df = pd.read_csv(DATA_RAW_PATH, nrows=100000)

print(df)

C:\Users\Rujuta\AppData\Local\Temp\ipykernel_48312\3276199243.py:6: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_RAW_PATH, nrows=100000)


             id  member_id  loan_amnt  funded_amnt  funded_amnt_inv  \
0      68407277        NaN     3600.0       3600.0           3600.0   
1      68355089        NaN    24700.0      24700.0          24700.0   
2      68341763        NaN    20000.0      20000.0          20000.0   
3      66310712        NaN    35000.0      35000.0          35000.0   
4      68476807        NaN    10400.0      10400.0          10400.0   
...         ...        ...        ...          ...              ...   
99995  62197780        NaN    20000.0      20000.0          20000.0   
99996  61374218        NaN     3000.0       3000.0           3000.0   
99997  60903843        NaN    14000.0      14000.0          14000.0   
99998  61973344        NaN    30000.0      30000.0          30000.0   
99999  62297713        NaN    24000.0      24000.0          24000.0   

             term  int_rate  installment grade sub_grade  \
0       36 months     13.99       123.03     C        C4   
1       36 months     11.99

### Dataset Size and Complexity

The accepted loans dataset contains approximately 151 columns, reflecting borrower attributes, loan terms, repayment behavior, and post-origination outcomes. Not all columns are suitable for modeling, and careful selection is required to avoid data leakage.


### Target Definition
In a loan / BNPL system, the decision is made at the time of application.

At that moment, the bank does not know the real outcome; it only has an estimate.

The actual outcome (repayment or default) is known only after time has passed, once the loan has been given.

When training a model, we learn from historical loans where the outcome is already known.

Therefore, the target variable must represent the real final outcome of past loans, because that reflects true user behavior and allows the model to learn meaningful patterns.

### Data Leakage
Any information created after the loan decision is made is not available at application time.

If such information is shown to the model during prediction, it breaks the real-world timeline.

Information like the final status of the loan directly reveals the outcome.

If the model already knows the outcome, it is no longer predicting anything.

Therefore, any information that directly or indirectly reveals the final result must never be available at prediction time.

In [10]:
columns_list = list(df_sample.columns)
for column in columns_list:
    print(column)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

### Summary
Target:
The target represents the final outcome of a loan. In this project, the target is loan_status, which indicates whether the user eventually repaid the loan or defaulted.

Leakage:
Leakage includes any columns that should not be available to the model at the time of prediction. This includes information related to the final loan status, payment-related information, and default-related information, since these are only known after the loan has been issued.

Simple rule to avoid leakage:
Any information that would not be known at the time a user applies for a loan should not be given to the model during prediction.

### Feature Availability Filtering 

### Columns to Exclude 
- loan_status
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- last_pymnt_d
- last_pymnt_amnt
- next_pymnt_d
- settlement_status
- settlement_date
- settlement_amount
- settlement_percentage
- settlement_term

This is an initial, conservative list of leakage columns and will be refined further during feature engineering and model evaluation.


### Candidate Features Available at Application Time


In [5]:
df.shape

(100000, 151)

In [6]:
df.head(50)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
5,68426831,NaN,11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
6,68476668,NaN,20000.0,20000.0,20000.0,36 months,9.17,637.58,B,B2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
7,67275481,NaN,20000.0,20000.0,20000.0,36 months,8.49,631.26,B,B1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
8,68466926,NaN,10000.0,10000.0,10000.0,36 months,6.49,306.45,A,A2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
9,68616873,NaN,8000.0,8000.0,8000.0,36 months,11.48,263.74,B,B5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:

df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='object', length=151)

In [13]:
cols = list(df.columns)
cols

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [14]:
df['loan_status'].value_counts()

loan_status
Fully Paid            70288
Charged Off           17603
Current               11402
Late (31-120 days)      441
In Grace Period         199
Late (16-30 days)        66
Default                   1
Name: count, dtype: int64

In [16]:
loan_status_list = ['Fully Paid', 'Charged Off', 'Default']
df_filtered = df[df['loan_status'].isin(loan_status_list)]

df_filtered['loan_status'].value_counts()

loan_status
Fully Paid     70288
Charged Off    17603
Default            1
Name: count, dtype: int64